In [2]:
%load_ext autoreload
%autoreload 2

import src.SHAP_like_graph_tool as gp

import networkx as nx
import html
import io
import json
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import itertools

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Lancement pour Graphes .graphml

In [3]:
def load_graphml_safe(path):
        with open(path, 'r', encoding='utf-8') as f:
            raw_data = f.read()

        clean_data = html.unescape(raw_data)
        G = nx.read_graphml(io.StringIO(clean_data))
        
        print(f"✅ Graphe chargé : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
        return G

In [4]:
import graph_tool.all as gt

name = "faa_routes"
#name = "eu_airlines"

print("📥 Chargement du graphe depuis la collection...")
G = gt.collection.ns[name]
print(G.list_properties())

G.edge_properties.clear()
G.graph_properties.clear()
print("Nettoyage terminé. Propriétés restantes :", G.list_properties())
G.set_directed(False)
gt.remove_parallel_edges(G)
gt.remove_self_loops(G)
G.purge_vertices() 

print(f"✅ Graphe prêt : {G.num_vertices()} nœuds, {G.num_edges()} arêtes (non-dirigé).")


prop_pos = G.vertex_properties["_pos"]

# 2. Conversion vers NetworkX
G_nx = nx.Graph()

for v in G.vertices():
    v_id = str(int(v)) # ID en string pour la conformité GraphML standard
    coords = list(prop_pos[v]) 
    pos_as_str = json.dumps(coords)
    G_nx.add_node(v_id, _pos=pos_as_str)

for e in G.edges():
    G_nx.add_edge(str(int(e.source())), str(int(e.target())))

# 3. Exportation
file_path = f"graph_library/benchmark_graphes_reels/reel_spatial_{name}.graphml"
nx.write_graphml(G_nx, file_path, named_key_ids=True)

print(f"✅ Graphe converti. Nœud exemple : {G_nx.nodes[str(int(G.vertex(0)))]}")

G_nx_test = load_graphml_safe(file_path)




📥 Chargement du graphe depuis la collection...
name              (graph)   (type: string, val: faa_routes)
description       (graph)   (type: string, val: A network of air traffic
                                                routes, from the FAA (Federal
                                                Aviation Administration)
                                                National Flight Data Center
                                                (NFDC) preferred routes
                                                database (www.fly.faa.gov).
                                                Date of extraction is prior to
                                                2010. Nodes represent airports
                                                or service centers, and a
                                                directed edge is the preferred
                                                route between airport i and
                                                airport j.

In [ ]:
G_name_list = ["reel_blumenau_drug"]

for G_name in G_name_list : 
    G = load_graphml_safe(f"graph_library/{G_name}.graphml")
    gp.execute(G, G_name)

In [ ]:
G_name = "reel_Airports"

G_train, data_train, _, _,_,_ = gp.load_all_data_for_graph(G_name)
all_densities = gp.prepare_all_densities(G_train)

communities = nx.get_node_attributes(G_train, 'sbm_id')
counts = {}

for node, comm in communities.items():
    counts[comm] = counts.get(comm, 0) + 1

print(counts)

prob_matrix = all_densities['sbm']

import matplotlib.pyplot as plt

# 1. Calcul des données relatives
total_nodes = G_train.number_of_nodes()
# On trie par taille décroissante pour le graphique également
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
labels = [f"Comm {c}" for c, s in sorted_counts]
percentages = [(s / total_nodes) * 100 for c, s in sorted_counts]

# 2. Affichage textuel (votre code)
print("Récapitulatif des tailles :")
for comm, size in sorted_counts:
    print(f"Communauté {comm} : {size/total_nodes:.2%} % des nœuds ({size} nœuds)")

print(f"\nTotal : {total_nodes} nœuds")

# 3. Visualisation de la distribution relative
plt.figure(figsize=(12, 6))
bars = plt.bar(labels, percentages, color='teal', edgecolor='black', alpha=0.8)

# Ajouter les pourcentages au-dessus des barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

plt.title(f"Distribution relative des communautés - {G_name}", fontsize=14)
plt.ylabel("Pourcentage du total des nœuds (%)", fontsize=12)
plt.xlabel("Communautés (triées par taille)", fontsize=12)
plt.ylim(0, max(percentages) * 1.15) # Laisser de la place pour les labels
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 2 - Lancement pour graphes sbm_ratio

In [ ]:
list_i = [1.00, 0.00]

for i in np.arange(0.00, 1.25, 0.25):
#for i in list_i:
    print("######################################")
    print(f"#### graph sbm {i:.2f} pos {1-i:.2f} ####")
    print("######################################")
    G_name = f"artificial_graph_sbmv2_{f'{i:.2f}'.replace('.', '_')}_pos_{f'{1-i:.2f}'.replace('.', '_')}"
    print(G_name)

    with open(f"graph_library/{G_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)  

    try:
        G = nx.node_link_graph(data)
        print(f"Graphe chargé avec succès : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
    except Exception as e:
        print(f"Erreur lors de la conversion : {e}")

    gp.execute(G_name)

In [ ]:
%load_ext autoreload
%autoreload 2

import src.SHAP_like_graph_tool as gp

import networkx as nx
import html
import io
import json
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import itertools

## Lancement pour Graphes .graphml

In [ ]:
def load_graphml_safe(path):
        with open(path, 'r', encoding='utf-8') as f:
            raw_data = f.read()

        clean_data = html.unescape(raw_data)
        G = nx.read_graphml(io.StringIO(clean_data))
        
        print(f"✅ Graphe chargé : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
        return G

In [ ]:
G_name_list = ["reel_facebook_friends"]

for G_name in G_name_list : 
    G = load_graphml_safe(f"graph_library/{G_name}.graphml")
    G_train_with_structure = gp.computeStructureFeatures(G)
    G_train_with_communities = gp.computeCommunityFeatures(G_train_with_structure)
    print("fini! tot")
    G_train_with_distances = gp.computeDistanceFeatures(G_train_with_communities)
    print("fini!")
    gp.loadsave_data_joblib(data=G_train_with_distances, filename=f"G_train_w_struct_com_dist_{G_name}", mode="save", talk=True)

In [ ]:
G_name = "reel_Airports"

G_train, data_train, _, _,_,_ = gp.load_all_data_for_graph(G_name)
all_densities = gp.prepare_all_densities(G_train)

communities = nx.get_node_attributes(G_train, 'sbm_id')
counts = {}

for node, comm in communities.items():
    counts[comm] = counts.get(comm, 0) + 1

print(counts)

prob_matrix = all_densities['sbm']

import matplotlib.pyplot as plt

# 1. Calcul des données relatives
total_nodes = G_train.number_of_nodes()
# On trie par taille décroissante pour le graphique également
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
labels = [f"Comm {c}" for c, s in sorted_counts]
percentages = [(s / total_nodes) * 100 for c, s in sorted_counts]

# 2. Affichage textuel (votre code)
print("Récapitulatif des tailles :")
for comm, size in sorted_counts:
    print(f"Communauté {comm} : {size/total_nodes:.2%} % des nœuds ({size} nœuds)")

print(f"\nTotal : {total_nodes} nœuds")

# 3. Visualisation de la distribution relative
plt.figure(figsize=(12, 6))
bars = plt.bar(labels, percentages, color='teal', edgecolor='black', alpha=0.8)

# Ajouter les pourcentages au-dessus des barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

plt.title(f"Distribution relative des communautés - {G_name}", fontsize=14)
plt.ylabel("Pourcentage du total des nœuds (%)", fontsize=12)
plt.xlabel("Communautés (triées par taille)", fontsize=12)
plt.ylim(0, max(percentages) * 1.15) # Laisser de la place pour les labels
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 2 - Lancement pour graphes sbm_ratio

In [ ]:
list_i = [1.00, 0.00]

for i in np.arange(0.00, 1.25, 0.25):
#for i in list_i:
    print("######################################")
    print(f"#### graph sbm {i:.2f} pos {1-i:.2f} ####")
    print("######################################")
    G_name = f"artificial_graph_sbmv2_{f'{i:.2f}'.replace('.', '_')}_pos_{f'{1-i:.2f}'.replace('.', '_')}"
    print(G_name)

    with open(f"graph_library/{G_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)  

    try:
        G = nx.node_link_graph(data)
        print(f"Graphe chargé avec succès : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
    except Exception as e:
        print(f"Erreur lors de la conversion : {e}")

    gp.execute(G, G_name, "shap")